# Corizo Artificial Intelligence Internship
## Project 2: Spotify Songs' Genre Segmentation & Recommendation Support System

**Author / Intern:** Hamid Kamal  
**Role:** Data Science & Machine Learning Engineer Intern  
**Dataset:** Spotify Audio Features Dataset (`upload_efc975f9-7e96-4804-8ae1-691bc3670af7.csv`) — 32,833 tracks  
**Objective:** Complete end-to-end data preprocessing, exploratory data analysis (EDA), correlation profiling, feature standardization, PCA dimensionality reduction, unsupervised K-Means clustering ($K=6$), cluster archetype profiling, and content-based recommendation prototype.


## 1. Environment Setup & Dependency Imports

We import standard libraries for numerical computation, data wrangling, visualization, machine learning, and distance metrics. Random seeds are set for reproducibility.


In [ ]:
import os
import sys
import tempfile
import warnings
from pathlib import Path

# Environment & thread configuration for stability & reproducibility
os.environ['MPLCONFIGDIR'] = tempfile.gettempdir()
os.environ['LOKY_MAX_CPU_COUNT'] = '1'
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['OPENBLAS_NUM_THREADS'] = '1'
sys.stdout.reconfigure(encoding='utf-8')
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score
from sklearn.metrics.pairwise import cosine_similarity

# Styling setup
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.edgecolor'] = '#cccccc'
plt.rcParams['axes.linewidth'] = 0.8

RANDOM_STATE = 42

# Candidate dataset paths (resolves from notebooks/ or repo root)
DATASET_CANDIDATES = [
    Path('../data/upload_efc975f9-7e96-4804-8ae1-691bc3670af7.csv'),
    Path('../upload_efc975f9-7e96-4804-8ae1-691bc3670af7.csv'),
    Path('data/upload_efc975f9-7e96-4804-8ae1-691bc3670af7.csv'),
    Path('upload_efc975f9-7e96-4804-8ae1-691bc3670af7.csv')
]

DATASET_PATH = None
for p in DATASET_CANDIDATES:
    if p.exists():
        DATASET_PATH = p
        break

if DATASET_PATH is None:
    DATASET_PATH = Path('upload_efc975f9-7e96-4804-8ae1-691bc3670af7.csv')

print(f"Environment configured. Dataset path resolved to: {DATASET_PATH}")


## 2. Dataset Ingestion & Initial Audit

We load the raw CSV dataset and inspect dimensions, column schema, missing values, duplicate tracks, and category distributions.


In [ ]:
# Ingest dataset
df_raw = pd.read_csv(DATASET_PATH)

print(f"Total Rows Ingested: {df_raw.shape[0]:,}")
print(f"Total Columns: {df_raw.shape[1]}")
print(f"Unique Track IDs: {df_raw['track_id'].nunique():,}")
print(f"Unique Artists: {df_raw['track_artist'].nunique():,}")
print(f"Unique Playlists: {df_raw['playlist_name'].nunique():,}")
print(f"Exact Duplicate Rows: {df_raw.duplicated().sum()}")
print(f"Track IDs across multiple playlists: {df_raw.duplicated(subset=['track_id']).sum():,}")

# Missing values check
null_summary = df_raw.isnull().sum()
display(null_summary[null_summary > 0].to_frame(name='Missing Values'))

df_raw.head(3)


## 3. Data Preprocessing & Feature Engineering

### Preprocessing Actions:
1. **Handling Missing Values:** Exactly 5 rows (<0.015%) lack metadata (`track_name`, `track_artist`, `track_album_name`). These 5 rows are dropped, retaining 32,828 clean records.
2. **Duplicate Track Occurrences:** Songs appearing in multiple playlists are retained at the playlist level to map playlist tags to unsupervised clusters.
3. **Unit Conversion:** Convert `duration_ms` to `duration_min` for intuitive interpretation.
4. **Temporal Extraction:** Parse `track_album_release_date` to extract numeric `release_year`.
5. **Feature Segregation:** Metadata identifiers and category labels (`track_id`, `track_name`, `artist`, `album_id`, `playlist_id`, `playlist_name`, `genre`, `subgenre`) are excluded from clustering input to prevent artificial categorical bias.
6. **Feature Scaling:** Standardize the 10 continuous audio features using `StandardScaler` ($z = (x - \mu)/\sigma$).


In [ ]:
# 1. Drop 5 null metadata records
df_clean = df_raw.dropna(subset=['track_name', 'track_artist', 'track_album_name']).copy()
df_clean.reset_index(drop=True, inplace=True)

# 2. Conversions
df_clean['duration_min'] = df_clean['duration_ms'] / 60000.0
df_clean['release_year'] = pd.to_datetime(df_clean['track_album_release_date'], errors='coerce').dt.year

# 3. Select 10 continuous audio features
audio_features = [
    'danceability', 'energy', 'loudness', 'speechiness',
    'acousticness', 'instrumentalness', 'liveness', 'valence',
    'tempo', 'duration_min'
]

# 4. Standardize features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_clean[audio_features])
scaled_df = pd.DataFrame(X_scaled, columns=audio_features)

print(f"Cleaned dataset rows: {len(df_clean):,}")
print(f"Clustering matrix X_scaled shape: {X_scaled.shape}")
scaled_df.describe().round(3).T[['mean', 'std', 'min', '50%', 'max']]


## 4. Exploratory Data Analysis & Visualizations

We examine the distributions of genres, subgenres, popularity, and all audio characteristics.


In [ ]:
# 1. Genre and Subgenre Distributions
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

genre_counts = df_clean['playlist_genre'].value_counts()
sns.barplot(x=genre_counts.values, y=genre_counts.index, hue=genre_counts.index, palette='viridis', ax=axes[0], legend=False)
axes[0].set_title('Distribution of Songs by Playlist Genre', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Number of Songs')
for i, v in enumerate(genre_counts.values):
    axes[0].text(v + 40, i, f"{v:,} ({v/len(df_clean)*100:.1f}%)", va='center', fontweight='bold', fontsize=9)

subgenre_counts = df_clean['playlist_subgenre'].value_counts()
sns.barplot(x=subgenre_counts.values, y=subgenre_counts.index, hue=subgenre_counts.index, palette='magma', ax=axes[1], legend=False)
axes[1].set_title('Distribution of Songs by Playlist Subgenre (24 Subgenres)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Number of Songs')

plt.tight_layout()
plt.show()


In [ ]:
# 2. Histograms & KDE Distributions of Audio Features & Popularity
fig, axes = plt.subplots(3, 4, figsize=(18, 12))
axes = axes.flatten()
plot_cols = audio_features + ['track_popularity', 'release_year']
colors = sns.color_palette('tab10', len(plot_cols))

for idx, col in enumerate(plot_cols):
    data = df_clean[col].dropna()
    sns.histplot(data, kde=True, ax=axes[idx], color=colors[idx], bins=30)
    axes[idx].set_title(f'Distribution of {col}', fontweight='bold', fontsize=11)
    axes[idx].set_ylabel('Frequency')
    mean_val = data.mean()
    median_val = data.median()
    axes[idx].axvline(mean_val, color='red', linestyle='--', linewidth=1.2, label=f'Mean: {mean_val:.2f}')
    axes[idx].axvline(median_val, color='green', linestyle=':', linewidth=1.2, label=f'Median: {median_val:.2f}')
    axes[idx].legend(fontsize=8)

for j in range(len(plot_cols), len(axes)):
    fig.delaxes(axes[j])

plt.suptitle('Distributions of Spotify Audio Features, Popularity, and Release Year', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()


In [ ]:
# 3. Audio Characteristics Across Playlist Genres
genre_means = df_clean.groupby('playlist_genre')[audio_features + ['track_popularity']].mean()

fig, axes = plt.subplots(3, 3, figsize=(16, 12))
axes = axes.flatten()
features_to_compare = ['danceability', 'energy', 'loudness', 'speechiness', 'acousticness', 'instrumentalness', 'valence', 'tempo', 'track_popularity']

for idx, feat in enumerate(features_to_compare):
    order = genre_means[feat].sort_values(ascending=False).index
    sns.barplot(data=df_clean, x='playlist_genre', y=feat, order=order, hue='playlist_genre', palette='Blues_r', ax=axes[idx], legend=False, errorbar=None)
    axes[idx].set_title(f'Average {feat.capitalize()} by Genre', fontweight='bold', fontsize=11)
    axes[idx].set_xlabel('')
    axes[idx].set_ylabel(feat)
    for p in axes[idx].patches:
        axes[idx].annotate(f"{p.get_height():.2f}", (p.get_x() + p.get_width() / 2., p.get_height()),
                           ha='center', va='bottom', fontsize=9, xytext=(0, 2), textcoords='offset points')

plt.suptitle('Comparison of Audio Characteristics Across Playlist Genres', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()


## 5. Correlation Analysis

We calculate the Pearson correlation matrix for all 10 audio features and track popularity:
- **Strong Positive Correlation:** `energy` and `loudness` ($r = +0.6767$) show that high-energy tracks are mastered at higher volume levels.
- **Strong Negative Correlations:** `energy` vs `acousticness` ($r = -0.5397$) and `loudness` vs `acousticness` ($r = -0.3616$).
- **Moderate Positivity Correlation:** `danceability` and `valence` ($r = +0.3305$).
- **Popularity Independence:** `track_popularity` has very low linear correlation ($|r| \le 0.15$) with any single audio feature.


In [ ]:
# Correlation Matrix
corr_features = audio_features + ['track_popularity']
corr_matrix = df_clean[corr_features].corr()

plt.figure(figsize=(10, 8))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
cmap = sns.diverging_palette(230, 20, as_cmap=True)

sns.heatmap(corr_matrix, mask=mask, cmap=cmap, vmin=-0.7, vmax=0.7, annot=True, fmt='.2f',
            square=True, linewidths=.5, cbar_kws={"shrink": .8})
plt.title('Correlation Matrix of Spotify Audio Features & Track Popularity', fontsize=13, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()


## 6. Dimensionality Reduction (PCA)

Principal Component Analysis (PCA) is applied to standardized audio features for variance assessment and 2D visualization.


In [ ]:
# Apply PCA
pca = PCA()
X_pca = pca.fit_transform(X_scaled)
evr = pca.explained_variance_ratio_
cum_evr = np.cumsum(evr)

pca_df_summary = pd.DataFrame({
    'Principal Component': [f'PC{i}' for i in range(1, len(evr)+1)],
    'Explained Variance Ratio': evr,
    'Explained Variance (%)': evr * 100,
    'Cumulative Variance (%)': cum_evr * 100
})
display(pca_df_summary.round(3))

# Scree & Cumulative Variance Plot
plt.figure(figsize=(10, 5))
plt.plot(range(1, len(evr)+1), cum_evr * 100, marker='o', linestyle='-', color='#1DB954', label='Cumulative Variance Explained', linewidth=2)
plt.bar(range(1, len(evr)+1), evr * 100, alpha=0.6, color='#2E77D0', label='Individual Variance Explained')
plt.xlabel('Principal Component Index', fontsize=11, fontweight='bold')
plt.ylabel('Explained Variance (%)', fontsize=11, fontweight='bold')
plt.title('PCA Explained Variance & Scree Plot', fontsize=13, fontweight='bold')
plt.xticks(range(1, len(evr)+1))
plt.axhline(y=70, color='red', linestyle='--', alpha=0.7, label='70% Threshold')
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# 2D PCA Space colored by Genre
pca_df = pd.DataFrame(X_pca[:, :2], columns=['PC1', 'PC2'])
pca_df['playlist_genre'] = df_clean['playlist_genre']

plt.figure(figsize=(11, 8))
sns.scatterplot(data=pca_df, x='PC1', y='PC2', hue='playlist_genre', alpha=0.35, s=20, palette='Set1')
plt.title('Spotify Songs in 2D PCA Space Colored by Playlist Genre', fontsize=13, fontweight='bold')
plt.xlabel(f'PC1 ({evr[0]*100:.1f}% Variance - Energy/Loudness vs Acousticness)', fontsize=11)
plt.ylabel(f'PC2 ({evr[1]*100:.1f}% Variance - Danceability/Valence vs Tempo)', fontsize=11)
plt.legend(title='Playlist Genre', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()


## 7. Determining the Number of Clusters (K)

### Multi-Metric Evaluation & Selection Trade-Offs:
- $K=2$ produces the highest overall mathematical silhouette score ($0.1777$), but splits the dataset into only two broad macro-groups (loud/electronic vs quiet/acoustic).
- Among granular multi-cluster candidates ($K \ge 3$), $K=6$ reaches the highest silhouette score ($0.1379$).
- While higher cluster counts ($K=8..10$) achieve lower Davies-Bouldin values ($1.7092..1.6374$), they lead to smaller, over-fragmented clusters.
- $K=6$ balances elbow flattening, multi-metric performance, and domain interpretability.


In [ ]:
# Evaluate K from 2 to 10
k_range = range(2, 11)
inertias = []
sil_scores = []
ch_scores = []
db_scores = []

np.random.seed(RANDOM_STATE)
sample_idx = np.random.choice(len(X_scaled), size=10000, replace=False)

for k in k_range:
    km_test = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    labels = km_test.fit_predict(X_scaled)
    inertias.append(km_test.inertia_)
    sil = silhouette_score(X_scaled[sample_idx], labels[sample_idx])
    sil_scores.append(sil)
    ch = calinski_harabasz_score(X_scaled, labels)
    ch_scores.append(ch)
    db = davies_bouldin_score(X_scaled, labels)
    db_scores.append(db)

eval_df = pd.DataFrame({
    'K': list(k_range),
    'Inertia (WCSS)': inertias,
    'Silhouette Score': sil_scores,
    'Calinski-Harabasz': ch_scores,
    'Davies-Bouldin': db_scores
})
display(eval_df.round(4))

# Elbow & Silhouette Curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(k_range, inertias, marker='o', color='#1DB954', linewidth=2, markersize=7)
axes[0].set_title('Elbow Method (Inertia vs K)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Number of Clusters (K)', fontsize=11)
axes[0].set_ylabel('Within-Cluster Sum of Squares (Inertia)', fontsize=11)
axes[0].set_xticks(k_range)
axes[0].axvline(x=6, color='red', linestyle='--', alpha=0.8, label='Selected K=6')
axes[0].legend()

axes[1].plot(k_range, sil_scores, marker='s', color='#2E77D0', linewidth=2, markersize=7)
axes[1].set_title('Silhouette Score vs K', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Number of Clusters (K)', fontsize=11)
axes[1].set_ylabel('Silhouette Score', fontsize=11)
axes[1].set_xticks(k_range)
axes[1].axvline(x=6, color='red', linestyle='--', alpha=0.8, label='Selected K=6 (Peak for K>=3)')
axes[1].legend()

plt.suptitle('Evaluation of Clustering Metrics across K', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


## 8. Final K-Means Model Training & Cluster Interpretation

We fit K-Means with $K=6$ ($n\_init=20$, $random\_state=42$) and assign descriptive acoustic profile labels.


In [ ]:
# Fit Final K-Means
k_optimal = 6
kmeans_model = KMeans(n_clusters=k_optimal, random_state=RANDOM_STATE, n_init=20)
df_clean['cluster'] = kmeans_model.fit_predict(X_scaled)
pca_df['cluster'] = df_clean['cluster']

cluster_names = {
    0: "Cluster 0: Upbeat Pop & Dance Anthems",
    1: "Cluster 1: Live & High-Energy Concerts",
    2: "Cluster 2: Rhythmic Rap & Urban Beats",
    3: "Cluster 3: Acoustic & Melodic Soul",
    4: "Cluster 4: Instrumental & Synth Soundscapes",
    5: "Cluster 5: Intense Fast-Tempo EDM & Rock"
}
df_clean['cluster_profile'] = df_clean['cluster'].map(cluster_names)
pca_df['cluster_profile'] = df_clean['cluster_profile']

print("Discovered Cluster Sizes & Proportions:")
for name, cnt in df_clean['cluster_profile'].value_counts().items():
    print(f"  {name:<46}: {cnt:5,d} songs ({cnt/len(df_clean)*100:5.1f}%)")

# Mean Audio Feature Profiles per Cluster
cluster_means = df_clean.groupby('cluster')[audio_features].mean()
display(cluster_means.round(3).T)


In [ ]:
# Heatmap of Cluster Profiles
plt.figure(figsize=(12, 6))
sns.heatmap(cluster_means.T, annot=True, fmt='.3f', cmap='YlGnBu', linewidths=.5,
            yticklabels=[f.capitalize() for f in audio_features],
            xticklabels=[f"Cluster {i}" for i in range(k_optimal)])
plt.title('Mean Unscaled Audio Feature Profiles per Discovered Cluster', fontsize=13, fontweight='bold', pad=15)
plt.xlabel('Cluster ID', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()

# 2D PCA Cluster Segmentation Plot with Centroids
plt.figure(figsize=(12, 8))
pca_centroids = pca.transform(kmeans_model.cluster_centers_)[:, :2]
sns.scatterplot(data=pca_df, x='PC1', y='PC2', hue='cluster_profile', alpha=0.4, s=25, palette='tab10')
plt.scatter(pca_centroids[:, 0], pca_centroids[:, 1], c='black', s=220, marker='X', edgecolors='white', linewidths=2, label='Cluster Centroids', zorder=5)
plt.title('K-Means Audio Segmentation in 2D PCA Space (K=6)', fontsize=13, fontweight='bold')
plt.xlabel(f'PC1 ({evr[0]*100:.1f}% Variance)', fontsize=11)
plt.ylabel(f'PC2 ({evr[1]*100:.1f}% Variance)', fontsize=11)
plt.legend(title='Discovered Audio Cluster', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()


## 9. Playlist, Genre, and Subgenre Cross-Analysis

We explore the distribution of Spotify playlist genres, subgenres, and top playlist names across the 6 discovered clusters.


In [ ]:
# Genre vs Cluster Cross Tabulations
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

genre_cluster_ct = pd.crosstab(df_clean['cluster_profile'], df_clean['playlist_genre'], normalize='index') * 100
genre_cluster_ct.plot(kind='barh', stacked=True, colormap='Spectral', ax=axes[0], edgecolor='white')
axes[0].set_title('Playlist Genre Composition (%) within Each Cluster', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Percentage (%)')
axes[0].set_ylabel('')
axes[0].legend(title='Genre', bbox_to_anchor=(1.02, 1), loc='upper left')

subgenre_cluster_ct = pd.crosstab(df_clean['playlist_genre'], df_clean['cluster'], normalize='index') * 100
sns.heatmap(subgenre_cluster_ct, annot=True, fmt='.1f', cmap='Blues', ax=axes[1], cbar_kws={'label': '% of Genre in Cluster'})
axes[1].set_title('Distribution of Playlist Genres Across Clusters (%)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Cluster ID')
axes[1].set_ylabel('Playlist Genre')

plt.suptitle('Relationship Between Discovered Audio Clusters & Spotify Playlist Genres', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# Top 10 Playlists Cluster Distribution
top_playlists = df_clean['playlist_name'].value_counts().head(10).index
playlist_cluster_df = df_clean[df_clean['playlist_name'].isin(top_playlists)]
top_pl_ct = pd.crosstab(playlist_cluster_df['playlist_name'], playlist_cluster_df['cluster'], normalize='index') * 100
top_pl_ct.plot(kind='barh', stacked=True, colormap='tab10', edgecolor='white', figsize=(12, 6))
plt.title('Cluster Distribution Across Top 10 High-Volume Spotify Playlists', fontsize=12, fontweight='bold')
plt.xlabel('Proportion of Songs (%)')
plt.ylabel('Playlist Name')
plt.legend(title='Cluster ID', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()


## 10. Content-Based Recommendation System Demonstration

We implement a content-based recommendation prototype using **Cosine Similarity** over standardized audio feature vectors:
$$\text{Similarity}(\mathbf{u}, \mathbf{v}) = \frac{\mathbf{u} \cdot \mathbf{v}}{\|\mathbf{u}\|_2 \|\mathbf{v}\|_2}$$

*Note:* This is a prototype demonstrating vector similarity and clustering support; it is not a production-scale collaborative recommender.


In [ ]:
def recommend_spotify_tracks(track_title, n_recommendations=5):
    """
    Content-based music recommender using cosine similarity over standardized audio features.
    """
    matches = df_clean[df_clean['track_name'].str.lower().str.contains(track_title.lower(), na=False)]
    if len(matches) == 0:
        print(f"Track containing '{track_title}' not found.")
        return None
    
    idx = matches.index[0]
    target_row = df_clean.iloc[idx]
    target_vec = X_scaled[idx].reshape(1, -1)
    
    sim_scores = cosine_similarity(target_vec, X_scaled)[0]
    sim_df = df_clean.copy()
    sim_df['similarity_score'] = sim_scores
    
    filtered = sim_df[sim_df['track_name'].str.lower() != target_row['track_name'].lower()]
    unique_recs = (filtered.drop_duplicates(subset=['track_name', 'track_artist'])
                           .sort_values('similarity_score', ascending=False)
                           .head(n_recommendations))
    
    print("=" * 80)
    print(f"SEED QUERY: '{target_row['track_name']}' by {target_row['track_artist']}")
    print(f"Genre: {target_row['playlist_genre']} ({target_row['playlist_subgenre']}) | Cluster: {target_row['cluster_profile']}")
    print(f"Acoustics: Dance={target_row['danceability']:.2f}, Energy={target_row['energy']:.2f}, Valence={target_row['valence']:.2f}, Loudness={target_row['loudness']:.1f}dB, Tempo={target_row['tempo']:.0f}BPM")
    print("-" * 80)
    print(f"TOP {n_recommendations} CONTENT-BASED RECOMMENDATIONS:")
    for i, (_, r) in enumerate(unique_recs.iterrows(), 1):
        print(f"{i}. '{r['track_name']}' by {r['track_artist']}")
        print(f"   Similarity: {r['similarity_score']:.4f} | Genre: {r['playlist_genre']} ({r['playlist_subgenre']}) | Cluster: {r['cluster']}")
        print(f"   Acoustics: Dance={r['danceability']:.2f}, Energy={r['energy']:.2f}, Valence={r['valence']:.2f}, Loudness={r['loudness']:.1f}dB, Tempo={r['tempo']:.0f}BPM\n")
        
    return target_row, unique_recs

# Run 5 test queries across diverse genres
q1 = recommend_spotify_tracks('Shape of You', 5)
q2 = recommend_spotify_tracks('Lose Yourself to Dance', 5)
q3 = recommend_spotify_tracks('Bohemian Rhapsody', 5)
q4 = recommend_spotify_tracks('Animals', 5)
q5 = recommend_spotify_tracks('Stay With Me', 5)


## 11. Evaluation, Limitations & Conclusion

### Summary of Achievements:
- Preprocessed and standardized 32,828 songs across 10 continuous acoustic features.
- Evaluated $K=2..10$, establishing $K=6$ as a practical multi-cluster segmentation ($0.1379$ silhouette for $K \ge 3$).
- Defined 6 distinct, musically interpretable cluster archetypes.
- Developed a content-based recommendation prototype using cosine similarity vector search.
- Exported the clustered dataset to `data/spotify_clustered_dataset.csv`.
